In [1]:
import pandas as pd
import numpy as np
from google.colab import files

In [16]:
uploaded = files.upload()

Saving eprom.json to eprom (1).json
Saving promotions.csv to promotions.csv


In [17]:
df=pd.read_json("eprom.json")
ad=pd.read_csv("promotions.csv")

## 1. KHẢO SÁT DỮ LIỆU GỐC (DATA PROFILING)

In [4]:
# 1.1. Kiểm tra kích thước (shape), các cột, kiểu dữ liệu hiện tại
print("DataFrame Shape:", df.shape)
print("\nDataFrame Info:")
df.info()

DataFrame Shape: (1000, 10)

DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   promo_id             1000 non-null   object
 1   promo_name           999 non-null    object
 2   promo_type           1000 non-null   object
 3   discount_value       1000 non-null   object
 4   start_date           1000 non-null   object
 5   end_date             1000 non-null   object
 6   applicable_category  217 non-null    object
 7   promo_channel        1000 non-null   object
 8   stackable_flag       1000 non-null   int64 
 9   min_order_value      1000 non-null   int64 
dtypes: int64(2), object(8)
memory usage: 78.3+ KB


In [21]:
# 1.2. Thống kê tỷ lệ khuyết thiếu/null trên từng cột
print("\nMissing Values (Count and Percentage):")
missing_data = df.isnull().sum()
missing_percentage = 100 * missing_data / len(df)
missing_df = pd.DataFrame({'Missing Count': missing_data, 'Missing Percentage': missing_percentage})
display(missing_df[missing_df['Missing Count'] > 0].sort_values(by='Missing Count', ascending=False))


Missing Values (Count and Percentage):


,Missing Count,Missing Percentage
applicable_category,783,78.3
promo_name,1,0.1


In [22]:
# 1.3. Kiểm tra số lượng dòng trùng lặp (duplicates) và tính duy nhất của khóa chính (Primary Key)
print("\nNumber of duplicate rows:", df.duplicated().sum())

# Assuming 'promo_id' is the primary key based on the data provided
if 'promo_id' in df.columns:
    print("\nNumber of duplicate 'promo_id' values:", df['promo_id'].duplicated().sum())
    print("Is 'promo_id' unique (True if all values are unique):", df['promo_id'].is_unique)
else:
    print("Primary key 'promo_id' not found in DataFrame.")


Number of duplicate rows: 0

Number of duplicate 'promo_id' values: 0
Is 'promo_id' unique (True if all values are unique): True


In [23]:
# 1.4. Kiểm tra các giá trị phân loại (.unique()) và khoảng giá trị số học cơ bản
print("\nUnique values for categorical columns and descriptive statistics for numerical columns:")
for column in df.columns:
    if df[column].dtype == 'object' or df[column].nunique() < 20: # Heuristic for categorical/low cardinality columns
        print(f"\nColumn '{column}' (Unique values):\n{df[column].unique()[:20]}{'...' if df[column].nunique() > 20 else ''}")
    elif pd.api.types.is_numeric_dtype(df[column]):
        print(f"\nColumn '{column}' (Descriptive Statistics):\n{df[column].describe()}")


Unique values for categorical columns and descriptive statistics for numerical columns:

Column 'promo_id' (Unique values):
['PROMO-0039-0001' 'PROMO-0029-0002' 'PROMO-0015-0003' 'PROMO-0043-0004'
 'PROMO-0008-0005' 'PROMO-0021-0006' 'PROMO-0039-0007' 'PROMO-0019-0008'
 'PROMO-0023-0009' 'PROMO-0011-0010' 'PROMO-0011-0011' 'PROMO-0024-0012'
 'PROMO-0036-0013' 'PROMO-0040-0014' 'PROMO-0024-0015' 'PROMO-0003-0016'
 'PROMO-0022-0017' 'PROMO-0002-0018' 'PROMO-0024-0019' 'PROMO-0044-0020']...

Column 'promo_name' (Unique values):
['Fall Launch 2020' 'Fall Launch 2018' 'Urban Blowout 2015'
 'Fall Launch 2021' 'Mid-Year Sale 2014' 'Spring Sale 2017'
 'Fall Launch 2016' 'Fall Launch 2017' 'Spring Sale 2015'
 'Year-End Sale 2017' 'Rural Special 2019' 'Year-End Sale 2020'
 'Fall Launch 2013' 'Mid-Year Sale 2017' None 'Year-End Sale 2021'
 'Year-End Sale 2018' 'Mid-Year Sale 2020' 'Mid-Year Sale 2013'
 'Fall Launch 2019']...

Column 'promo_type' (Unique values):
['percentage' 'fixed']

Column 'd

2. LÀM SẠCH VÀ CHUẨN HÓA (CLEANING & TRANSFORMATION)

In [24]:


# 2.1 & 2.2. Chuẩn hóa kiểu dữ liệu và làm sạch văn bản

# Trước khi strip, ta lưu lại vị trí của các dòng null và rỗng để gán tên cụ thể
# Giả sử df vẫn giữ nguyên từ cell ivdTH1HEKcRR
mask_null = df['promo_name'].isnull()
mask_empty = df['promo_name'].astype(str).str.strip() == ''

# Chuẩn hóa văn bản chung
text_cols = df.select_dtypes(include=['object']).columns
for col in text_cols:
    df[col] = df[col].astype(str).str.strip()

# Ép kiểu ngày tháng
date_cols = ['start_date', 'end_date']
for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

# Xử lý missing values cho category
df['applicable_category'] = df['applicable_category'].replace(['nan', 'None', ''], np.nan).fillna('All Categories')
df['promo_type'] = df['promo_type'].replace(['nan', 'None', ''], np.nan).fillna('All Categories')

# ĐIỀU CHỈNH THEO YÊU CẦU: Gán tên cụ thể cho 2 dòng bị thiếu
df.loc[mask_null, 'promo_name'] = 'Mid-Year Sale 2013'
df.loc[mask_empty, 'promo_name'] = 'Fall Launch 2014'

# Các dòng 'Unknown' còn lại (nếu có) sẽ giữ nguyên hoặc xử lý tiếp
df['promo_name'] = df['promo_name'].replace(['nan', 'None', ''], 'Unknown Promotion')

print("Hoàn tất điều chỉnh tên khuyến mãi: Mid-Year Sale 2013 và Fall Launch 2014.")

Hoàn tất điều chỉnh tên khuyến mãi: Mid-Year Sale 2013 và Fall Launch 2014.


In [25]:
# 2.5. Xử lý các giá trị số không hợp lý (Số âm)
print("Đang kiểm tra và điều chỉnh các giá trị âm không hợp lý...")

# Chuyển đổi discount_value sang số để xử lý (xử lý cả các trường hợp như '30%%' hoặc 'twenty percent')
def clean_discount(val):
    try:
        if isinstance(val, str):
            val = val.replace('%%', '').strip()
            # Xử lý các trường hợp chữ nếu cần thiết, ở đây tạm thời ép kiểu số
            val = float(val)
        return abs(val) # Lấy giá trị tuyệt đối để sửa lỗi số âm
    except:
        return 0.0

if 'discount_value' in df.columns:
    df['discount_value'] = df['discount_value'].apply(clean_discount)
    print("  - Đã điều chỉnh các giá trị âm trong discount_value thành số dương.")

if 'min_order_value' in df.columns:
    df['min_order_value'] = df['min_order_value'].apply(lambda x: max(0, x))
    print("  - Đã đảm bảo min_order_value không nhỏ hơn 0.")

Đang kiểm tra và điều chỉnh các giá trị âm không hợp lý...
  - Đã điều chỉnh các giá trị âm trong discount_value thành số dương.
  - Đã đảm bảo min_order_value không nhỏ hơn 0.


In [26]:
# 2.3. Xử lý giá trị trống (Null/NaN)


# Handle missing values in 'promo_type' (assuming this is the 'category' column the user referred to)
if 'promo_type' in df.columns:
    df['promo_type'] = df['promo_type'].fillna('All Categories')
    print("  - Filled missing 'promo_type' with 'All Categories'.")



print("Null values have been handled implicitly through type coercion and explicitly for 'promo_type'.")

  - Filled missing 'promo_type' with 'All Categories'.
Null values have been handled implicitly through type coercion and explicitly for 'promo_type'.


In [27]:
# 2.4. Loại bỏ trùng lặp
initial_count = len(df)
df = df.drop_duplicates()
final_count = len(df)
print(f"Rows before: {initial_count}")
print(f"Rows after removing duplicates: {final_count}")
print(f"Total duplicates removed: {initial_count - final_count}")

Rows before: 1000
Rows after removing duplicates: 1000
Total duplicates removed: 0


## 3. KIỂM TRA TÍNH TOÀN VẸN (DATA QUALITY ASSURANCE)

In [32]:
# 3.0. Điều chỉnh logic ngày tháng (Sửa lỗi start_date > end_date)
mask = df['start_date'] > df['end_date']
if mask.any():
    print(f"Phát hiện {mask.sum()} dòng có start_date > end_date")
    df.loc[mask, ['start_date', 'end_date']] = df.loc[mask, ['end_date', 'start_date']].values
    print("Đã sửa lỗi logic ngày tháng.")

In [28]:
# 3.1. Kiểm tra Khóa chính (promo_id)
assert df['promo_id'].notnull().all(), "Lỗi: promo_id có giá trị null!"
assert df['promo_id'].is_unique, "Lỗi: promo_id bị trùng lặp!"
print("QA 3.1: Khóa chính hợp lệ.")

QA 3.1: Khóa chính hợp lệ.


In [33]:
# 3.2. Kiểm tra Logic thời gian (Sau khi đã sửa)
if 'start_date' in df.columns and 'end_date' in df.columns:
    date_logic = (df['start_date'] <= df['end_date']) | df['start_date'].isnull() | df['end_date'].isnull()
    assert date_logic.all(), "Lỗi: Vẫn còn dòng có start_date > end_date!"
print("QA 3.2: Logic thời gian hợp lệ.")

QA 3.2: Logic thời gian hợp lệ.


In [34]:
# 3.3. Kiểm tra ràng buộc số học
# Lưu ý: discount_value có thể chứa chuỗi như 'twenty percent', cần xử lý thêm nếu muốn kiểm tra số học triệt để
# Ở đây ta kiểm tra các giá trị đã là số
numeric_discount = pd.to_numeric(df['discount_value'], errors='coerce')
assert (numeric_discount.dropna() >= 0).all(), "Lỗi: discount_value có giá trị âm!"
print("QA 3.3: Ràng buộc số học hợp lệ.")

QA 3.3: Ràng buộc số học hợp lệ.


### 4. Tạo Bảng và Kiểm Tra Khóa Ngoại
Chúng ta sẽ tạo một DataFrame để mapping giữa `promo_id` của file `eprom` và file `promotions.csv`.

In [36]:


# promotion_id [PK] lấy từ promo_id của eprom (df)
# promo_id [FK] lấy từ promo_id của promotions (ad)

# Lấy danh sách ID duy nhất từ eprom
eprom_ids = df[['promo_id']].drop_duplicates().rename(columns={'promo_id': 'promotion_id'})

# Thực hiện Join để kiểm tra sự tồn tại trong file promotions (ad)
# Giả định promo_id trong ad là master key
mapping_df = eprom_ids.copy()
mapping_df['promo_id_fk'] = mapping_df['promotion_id'].apply(lambda x: x.split('-')[0] + '-' + x.split('-')[1] if '-' in x else x)

# 5.2. Kiểm tra khóa ngoại (Foreign Key Integrity Check)
# Kiểm tra xem các ID rút trích từ eprom có tồn tại trong ad hay không
master_ids = set(ad['promo_id'].unique())
eprom_base_ids = set(mapping_df['promo_id_fk'].unique())

missing_ids = eprom_base_ids - master_ids

print(f"Tổng số promotion_id trong EPROM: {len(mapping_df)}")
if not missing_ids:
    print(" Kiểm tra khóa ngoại thành công: Tất cả promo_id trong EPROM đều tồn tại trong danh mục promotions.")
else:
    print(f" Cảnh báo: Có {len(missing_ids)} ID không tìm thấy trong file master promotions!")
    print(f"Ví dụ các ID thiếu: {list(missing_ids)[:5]}")

display(mapping_df.head())

# Xuất file mapping
mapping_df.to_csv('eprom_promotion_mapping.csv', index=False)

Tổng số promotion_id trong EPROM: 1000
 Kiểm tra khóa ngoại thành công: Tất cả promo_id trong EPROM đều tồn tại trong danh mục promotions.


,promotion_id,promo_id_fk
0,PROMO-0039-0001,PROMO-0039
1,PROMO-0029-0002,PROMO-0029
2,PROMO-0015-0003,PROMO-0015
3,PROMO-0043-0004,PROMO-0043
4,PROMO-0008-0005,PROMO-0008
